In [30]:
# [STEP 1] Environment Setup & Data Loading
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

RAW_DATA_PATH = '../data/raw/hotel_bookings.csv' 
PROCESSED_DIR = '../data/processed/'              

# Create directory for processed results if it doesn't exist
if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)

# Load raw data and create a working copy to preserve the original state
if os.path.exists(RAW_DATA_PATH):
    df_raw = pd.read_csv(RAW_DATA_PATH)
    df = df_raw.copy()
    print(f"Dataset loaded. Initial Shape: {df.shape}")
else:
    print("Error: Please place 'hotel_bookings.csv' in the './data/raw/' folder.")

Dataset loaded. Initial Shape: (119390, 32)


In [31]:
# [STEP 2] Data Cleaning & Feature Extraction
df.columns = df.columns.str.strip() # Remove whitespace

# 1. Agent Logic: Create binary flag and drop raw ID
if 'agent' in df.columns:
    df['has_agent'] = df['agent'].notnull().astype(int)
    df.drop('agent', axis=1, inplace=True)

# 2. Company Logic: Create binary flag ONLY if column exists, then drop
if 'company' in df.columns:
    df['has_company'] = df['company'].notnull().astype(int)
    df.drop('company', axis=1, inplace=True) # Prevent KeyError by checking existence
else:
    # If already dropped in previous runs, ensure has_company exists with 0
    if 'has_company' not in df.columns:
        df['has_company'] = 0

# 3. Fill remaining missing values
df['children'] = df['children'].fillna(0)
df['country'] = df['country'].fillna('Unknown')

# 4. Filter invalid records
df['total_guests'] = df['adults'] + df['children'] + df['babies']
df = df[df['total_guests'] > 0]

print(f"Cleaning complete. Current records: {len(df)}")

Cleaning complete. Current records: 119210


In [32]:
# [STEP 3] Feature Engineering & Data Leakage Prevention
# 1. Create New Features
df['total_stay'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
df['is_family'] = ((df['children'] > 0) | (df['babies'] > 0)).astype(int)

# 2. Simplify Country Data (Reduce noise by keeping Top 10)
top_10_countries = df['country'].value_counts().nlargest(10).index
df['country'] = df['country'].apply(lambda x: x if x in top_10_countries else 'Other')

# 3. Drop Leakage Columns (Determined after the booking outcome)
# reservation_status is a direct indicator of the target is_canceled
leak_cols = ['reservation_status', 'reservation_status_date']
df.drop([c for c in leak_cols if c in df.columns], axis=1, inplace=True)

print("Feature engineering and leakage prevention applied.")

Feature engineering and leakage prevention applied.


In [33]:
# [STEP 4] One-Hot Encoding & Final Verification
# List all relevant categorical columns as per lead feedback
cat_cols = [
    'hotel', 'meal', 'market_segment', 'distribution_channel', 
    'deposit_type', 'customer_type', 'arrival_date_month', 'country',
    'reserved_room_type', 'assigned_room_type'
]

# Convert categorical features into numeric dummies
df_final = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Final Verification: Ensure no 'object' types remain for modeling
remaining_objects = df_final.select_dtypes(include=['object']).columns
if remaining_objects.empty:
    print(f"Final dataset prepared. All features are numeric. Total columns: {df_final.shape[1]}")
else:
    print(f"Warning: Objects remaining -> {list(remaining_objects)}")

Final dataset prepared. All features are numeric. Total columns: 83


In [34]:
# [STEP 5] Export Processed Data
# 1. Classification Data: Full processed dataset for 'is_canceled' prediction
clf_output = os.path.join(PROCESSED_DIR, 'hotel_bookings_clf.csv')
df_final.to_csv(clf_output, index=False)

# 2. Regression Data: Predict 'adr' for actual guests with strict filtering
# Apply ADR outlier filtering (0 < ADR < 500) only for regression target stability
df_reg = df_final[df_final['is_canceled'] == 0].copy()
df_reg = df_reg[(df_reg['adr'] > 0) & (df_reg['adr'] < 500)]

reg_output = os.path.join(PROCESSED_DIR, 'hotel_bookings_reg.csv')
df_reg.to_csv(reg_output, index=False)

print(f"Success! Model-ready files saved in: {os.path.abspath(PROCESSED_DIR)}")

Success! Model-ready files saved in: c:\Users\양수겸\Desktop\DS_TermP_Team1\data\processed
